# GARCH Volatility and Risk Workflow

Module: Financial Time Series

## Lesson summary

This lab turns conditional volatility theory into a modeling workflow. Students fit a GARCH-family model, inspect volatility persistence, compare distributional assumptions, and connect conditional volatility to one-step risk limits {cite}`engle1982autoregressive,bollerslev1986generalized,tsay2010analysis`.

## Learning objectives

By the end of this lab, students should be able to:

- explain volatility clustering in return series;
- fit a GARCH model with the `arch` library;
- interpret $\omega$, $\alpha$, and $\beta$;
- evaluate volatility persistence and long-run variance;
- understand why Student's t errors can be useful for heavy-tailed returns;
- connect conditional volatility forecasts with dynamic VaR.

## Model equations

The GARCH(1,1) recursion updates conditional variance from the previous shock and previous variance:

$$
r_t = \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t,
$$

$$
\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2.
$$

Persistence is summarized by $\alpha+\beta$. When $\alpha+\beta<1$, the long-run variance is:

$$
\bar{\sigma}^2 = \frac{\omega}{1-\alpha-\beta}.
$$

The one-step VaR bridge uses the conditional mean, conditional volatility, and a distributional quantile:

$$
\operatorname{VaR}_{\alpha,t+1}= -\left(\mu_{t+1}+\sigma_{t+1}q_\alpha\right).
$$

## Setup

In [ ]:
import numpy as np
import pandas as pd
from arch import arch_model
from scipy.stats import norm, t

from src.time_series_diagnostics import (
    garch_long_run_variance,
    garch_persistence,
    parametric_var,
)

## Simulated clustered returns

The synthetic series below has time-varying volatility, so it behaves more like financial returns than white noise with constant variance.

In [ ]:
rng = np.random.default_rng(11)
dates = pd.bdate_range("2023-01-02", periods=750)

omega = 0.02
alpha = 0.08
beta = 0.90
variance = np.empty(len(dates))
returns = np.empty(len(dates))
variance[0] = omega / (1 - alpha - beta)

for t_idx in range(1, len(dates)):
    shock = rng.standard_t(df=6)
    returns[t_idx] = np.sqrt(variance[t_idx - 1]) * shock
    variance[t_idx] = omega + alpha * returns[t_idx] ** 2 + beta * variance[t_idx - 1]

returns = pd.Series(returns, index=dates, name="return_pct")
returns.head()

## Fit GARCH with Gaussian errors

The `arch` package expects returns in percentage units in many examples, which also improves numerical stability {cite}`sheppard2024arch`.

In [ ]:
garch_normal = arch_model(
    returns,
    mean="Constant",
    vol="Garch",
    p=1,
    q=1,
    dist="normal",
)
garch_normal_result = garch_normal.fit(disp="off")
garch_normal_result.summary()

## Persistence

In [ ]:
params = garch_normal_result.params
alpha_hat = params["alpha[1]"]
beta_hat = params["beta[1]"]
omega_hat = params["omega"]

{
    "persistence": garch_persistence(alpha_hat, beta_hat),
    "long_run_variance": garch_long_run_variance(omega_hat, alpha_hat, beta_hat),
}

## Fit GARCH with Student's t errors

In [ ]:
garch_t = arch_model(
    returns,
    mean="Constant",
    vol="Garch",
    p=1,
    q=1,
    dist="StudentsT",
)
garch_t_result = garch_t.fit(disp="off")

pd.DataFrame(
    {
        "normal": {"aic": garch_normal_result.aic, "bic": garch_normal_result.bic},
        "student_t": {"aic": garch_t_result.aic, "bic": garch_t_result.bic},
    }
)

## One-step VaR bridge

In [ ]:
forecast = garch_t_result.forecast(horizon=1)
mean_forecast = float(forecast.mean.iloc[-1, 0])
vol_forecast = float(np.sqrt(forecast.variance.iloc[-1, 0]))
nu = float(garch_t_result.params["nu"])

var_99_normal = parametric_var(mean_forecast, vol_forecast, norm.ppf(0.01))
var_99_student = parametric_var(mean_forecast, vol_forecast, t.ppf(0.01, df=nu))

{
    "mean_forecast": mean_forecast,
    "volatility_forecast": vol_forecast,
    "nu": nu,
    "normal_1pct_var": var_99_normal,
    "student_t_1pct_var": var_99_student,
}

## Extension: asymmetric volatility

To capture different volatility responses to negative and positive shocks, use a GJR-GARCH specification {cite}`glosten1993relation`:

```python
arch_model(returns, mean="Constant", vol="Garch", p=1, o=1, q=1, dist="StudentsT")
```

The additional `gamma` term measures asymmetric response. A positive and statistically significant `gamma` means negative shocks increase conditional variance more than positive shocks of the same magnitude.

## Model limitations

- GARCH parameters can be unstable across regimes, especially around crisis periods or policy breaks.
- Gaussian errors often understate tail risk; Student's t errors help but still impose a fixed distributional shape.
- One-step volatility forecasts do not capture liquidity, jumps, or changes in the data-generating process.